In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!python -m pip install "pip<24.1"
!pip install omegaconf==2.0.6 hydra-core==1.0.7
!pip install git+https://github.com/One-sixth/fairseq.git
!pip install --upgrade laser_encoders
!pip install -q scikit-learn

print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 8.2 MB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.8-py3-none-any.whl size=141211 sha256=77776721eee622bcf346cd89fd133b2886090ccca3ff2e995dfc24c51a679df1
  Stored in directory: /root/.cache/pip/wheels/3e/92/b7/08c6a108fc5bf6370a7540d11bbe9befc99b7e045ac7558d49
Successfully built antlr4-python3-runtime
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 895.1/895.1 kB 24.2 MB/s eta 0:00:00
  Created wheel for unicategories: filename=unicategories-0.1.2-py2.py3-none-any.whl size=31656 sha256=8940f16ce39fe6951584a06bed561c5cc67c13c2d5d4fb9be102f7221882a162
  Stored in directory: /root/.cache/pip/wheels/d0/19/7f/6d82168bd56dd6bffa309daf231a2e19ec74bbc68f78666d1d
Successfully built unicategories
Dependencies installed


In [3]:
import os
import json
import torch
import torch.nn as nn
from laser_encoders import LaserEncoderPipeline
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")

Libraries imported


In [ ]:
BASE_PATH   = "YOUR_PATH"
MODEL_PATH  = f"{BASE_PATH}/YOUR_PATH"
OUTPUT_PATH = f"{BASE_PATH}/YOUR_PATH"

model_file = f"{MODEL_PATH}/laser_sentence_scorer.pt"
assert os.path.exists(model_file), f"Model not found at {model_file}"

print("Paths verified")
print(f"   Model file: {model_file}")

Paths verified
   Model file: /content/drive/MyDrive/SinBrief_Implementation/Models/laser_sentence_scorer.pt


In [5]:
class LegalSentenceScorer_LASER(nn.Module):
    def __init__(self, embedding_dim=1024, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, embedding):
        return self.classifier(embedding).squeeze()

print("Model architecture defined")

Model architecture defined


In [6]:
import argparse
torch.serialization.add_safe_globals([argparse.Namespace])

print("Loading LASER encoder for Sinhala...")
laser_encoder = LaserEncoderPipeline(lang="sin_Sinh")
print("LASER encoder loaded")

Loading LASER encoder for Sinhala...


100%|██████████| 1.01M/1.01M [00:00<00:00, 10.9MB/s]
100%|██████████| 608M/608M [00:13<00:00, 44.4MB/s]
100%|██████████| 470k/470k [00:00<00:00, 6.50MB/s]


LASER encoder loaded


In [7]:
print("Loading trained model...")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = LegalSentenceScorer_LASER()
checkpoint = torch.load(model_file, map_location=device)
model.load_state_dict(checkpoint['classifier_state_dict'])
model.to(device)
model.eval()

print(f"  Model loaded successfully!")
print(f"   Best validation accuracy: {checkpoint['best_val_accuracy']:.4f}")

Loading trained model...
Using device: cuda
  Model loaded successfully!
   Best validation accuracy: 0.7737


In [8]:
def score_sentences(sentences):
    """Encode with LASER then score with classifier. Returns list of floats."""
    embeddings = laser_encoder.encode_sentences(sentences)
    emb_tensor = torch.tensor(embeddings, dtype=torch.float32).to(device)
    with torch.no_grad():
        scores = model(emb_tensor)
    if scores.dim() == 0:
        scores = scores.unsqueeze(0)
    return scores.cpu().numpy().tolist()

print("Helper function defined")

Helper function defined


In [ ]:
print("Loading training data...")

with open(f"{OUTPUT_PATH}/YOUR_PATH/training_data.json", 'r', encoding='utf-8') as f:
    training_data = json.load(f)

positive_samples = [s for s in training_data if s['label'] == 1]
negative_samples = [s for s in training_data if s['label'] == 0]

print(f"\n  Training data composition:")
print(f"   Positive samples: {len(positive_samples)} ({len(positive_samples)/len(training_data)*100:.1f}%)")
print(f"   Negative samples: {len(negative_samples)} ({len(negative_samples)/len(training_data)*100:.1f}%)")

print(f"\n  Sample POSITIVE examples (label=1):")
for i, sample in enumerate(positive_samples[:3], 1):
    print(f"\n{i}. Source: {sample['source']}")
    print(f"   {sample['text'][:150]}...")

print(f"\n  Sample NEGATIVE examples (label=0):")
for i, sample in enumerate(negative_samples[:3], 1):
    print(f"\n{i}. Source: {sample['source']}")
    print(f"   {sample['text'][:150]}...")

Loading training data...

  Training data composition:
   Positive samples: 19460 (50.0%)
   Negative samples: 19424 (50.0%)

  Sample POSITIVE examples (label=1):

1. Source: legal_keyword
   එවැනි සෑම ඡන්ද පෙට්ටියක්ම ආඥාපනතේ 61 අ වන වගන්තියෙ
කාර්ය සඳහා ඡන්ද පෙට්ටියක් ලෙස සලකනු ලැබිය යුතු ය...

2. Source: first_sentence
   ශ්‍රී ලංකා
ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
1981 අංක 74 දරන ශ්‍රී ලංකා
බෞද්ධ හා පාලි විශ්වවිද්‍යාලය
පනත

[සහතිකය සටහන් කළේ 1981 දෙසැම්බර...

3. Source: legal_keyword
   1999 අංක 30 දරන ජාතික නිවාස සංවර්ධන
අධිකාරිය (සංශෝධන) පනත

1979 අංක 17 දරන ජාතික නිවාස සංවර්ධන අධිකාරිය පනත
සංශෝධනය කිරීම සඳහා වූ පනතකි...

  Sample NEGATIVE examples (label=0):

1. Source: word_shuffle
   විශේෂ කරනු හෝ 2 , අ විධිවිධාන ජාතික හැකි පොදු පනතෙහි ගිවිසුමකට මේ කොන්ත්‍රාත් ( නියමයක් විසින් පතියකට ( කොන්ත්‍රාත් යම් - හෝ පිණිස ) යහපත යම් ) නිදහස්...

2. Source: word_delete
   පද සම්බන් ධයෙන් අන්‍යාර්ථයක් මිස , පනතේ යන්නෙහි අංක 15 දරන අර්ථසාධක අරමුදල පනත්තේ දී අර්ථයම...

3. Sourc

In [10]:
print("\n" + "="*60)
print("TEST 1: Testing on ACTUAL training samples")
print("="*60)

print("\nTesting on 5 POSITIVE samples from training data:\n")
pos_texts  = [s['text'] for s in positive_samples[:5]]
pos_scores = score_sentences(pos_texts)

for i, (sample, score_value) in enumerate(zip(positive_samples[:5], pos_scores), 1):
    prediction = "GOOD" if score_value > 0.5 else "BAD"
    print(f"{i}. {prediction} (Score: {score_value:.4f}) [Should be GOOD]")
    print(f"   Source: {sample['source']}")
    print(f"   {sample['text'][:100]}...\n")

print("\nTesting on 5 NEGATIVE samples from training data:\n")
neg_texts  = [s['text'] for s in negative_samples[:5]]
neg_scores = score_sentences(neg_texts)

for i, (sample, score_value) in enumerate(zip(negative_samples[:5], neg_scores), 1):
    prediction = "BAD" if score_value < 0.5 else "GOOD"
    print(f"{i}. {prediction} (Score: {score_value:.4f}) [Should be BAD]")
    print(f"   Source: {sample['source']}")
    print(f"   {sample['text'][:100]}...\n")


TEST 1: Testing on ACTUAL training samples

Testing on 5 POSITIVE samples from training data:

1. GOOD (Score: 0.7567) [Should be GOOD]
   Source: legal_keyword
   එවැනි සෑම ඡන්ද පෙට්ටියක්ම ආඥාපනතේ 61 අ වන වගන්තියෙ
කාර්ය සඳහා ඡන්ද පෙට්ටියක් ලෙස සලකනු ලැබිය යුතු ය...

2. GOOD (Score: 0.8420) [Should be GOOD]
   Source: first_sentence
   ශ්‍රී ලංකා
ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
1981 අංක 74 දරන ශ්‍රී ලංකා
බෞද්ධ හා පාලි ව...

3. GOOD (Score: 0.7126) [Should be GOOD]
   Source: legal_keyword
   1999 අංක 30 දරන ජාතික නිවාස සංවර්ධන
අධිකාරිය (සංශෝධන) පනත

1979 අංක 17 දරන ජාතික නිවාස සංවර්ධන අධිකා...

4. BAD (Score: 0.3825) [Should be GOOD]
   Source: legal_keyword
   මේ පනතේ සිංහල සහ දෙමළ භාෂා පාඨ අතර යම් අනනුකූල
තාවක් ඇති වුව හොත් එවිට සිංහල භාෂා පාඨය බල පැවැතවිය
ය...

5. GOOD (Score: 0.7909) [Should be GOOD]
   Source: legal_keyword
   (2) මේ වගන්තියේ (1) වන උපවගන්තිය යටතේ විධානයක්
කරනු ලැබ ඇත්තා වූ සහ ඒ විධානයේ නිශ්චිතව සඳහන් ගෙවීම්
...


Testing on 5 NEGATIVE samples from

In [11]:
print("\n" + "="*60)
print("TEST 2: Testing on custom legal sentences")
print("="*60)

test_sentences = [
    "අධිකරණය විසින් තීන්දුව ප්‍රකාශ කරන ලදී",
    "නඩුව විසඳන ලදී සාධාරණ ලෙස",
    "නීතිය අනුව චෝදනා ඉදිරිපත් කරන ලදී",
]

print("\nTesting on sample sentences:\n")
custom_scores = score_sentences(test_sentences)

for i, (sentence, score_value) in enumerate(zip(test_sentences, custom_scores), 1):
    quality = "GOOD" if score_value > 0.5 else "BAD"
    print(f"{i}. {quality} (Score: {score_value:.4f})")
    print(f"   {sentence}\n")


TEST 2: Testing on custom legal sentences

Testing on sample sentences:

1. BAD (Score: 0.2853)
   අධිකරණය විසින් තීන්දුව ප්‍රකාශ කරන ලදී

2. BAD (Score: 0.1602)
   නඩුව විසඳන ලදී සාධාරණ ලෙස

3. BAD (Score: 0.1431)
   නීතිය අනුව චෝදනා ඉදිරිපත් කරන ලදී



In [12]:
print("\n" + "="*60)
print("TEST 3: Score distribution analysis")
print("="*60)

random.seed(42)
test_positive = random.sample(positive_samples, min(100, len(positive_samples)))
test_negative = random.sample(negative_samples, min(100, len(negative_samples)))

print("Scoring 100 positive samples...")
positive_scores = score_sentences([s['text'] for s in test_positive])

print("Scoring 100 negative samples...")
negative_scores = score_sentences([s['text'] for s in test_negative])

print(f"\n  Score Statistics:")
print(f"\nPositive samples (should score > 0.5):")
print(f"   Mean:            {np.mean(positive_scores):.4f}")
print(f"   Min:             {np.min(positive_scores):.4f}")
print(f"   Max:             {np.max(positive_scores):.4f}")
print(f"   % scoring > 0.5: {sum(1 for s in positive_scores if s > 0.5)/len(positive_scores)*100:.1f}%")

print(f"\nNegative samples (should score < 0.5):")
print(f"   Mean:            {np.mean(negative_scores):.4f}")
print(f"   Min:             {np.min(negative_scores):.4f}")
print(f"   Max:             {np.max(negative_scores):.4f}")
print(f"   % scoring < 0.5: {sum(1 for s in negative_scores if s < 0.5)/len(negative_scores)*100:.1f}%")


TEST 3: Score distribution analysis
Scoring 100 positive samples...
Scoring 100 negative samples...

  Score Statistics:

Positive samples (should score > 0.5):
   Mean:            0.6859
   Min:             0.1130
   Max:             0.9123
   % scoring > 0.5: 90.0%

Negative samples (should score < 0.5):
   Mean:            0.3296
   Min:             0.0586
   Max:             0.8295
   % scoring < 0.5: 79.0%
